![NWM](img/NWM.png)

# Analyze ParFlow-CONUS Outputs vs Observed Snow Water Equivalent (SWE)
Authors: Irene Garousi-Nejad (igarousi@cuahsi.org), Danielle Tijerina-Kreuzer (dtijerina@cuahsi.org)  
Last updated: Feb 2026

#### Introduction:  
This notebook demonstrates how to perform a point-scale analysis comparing modeled and observed SWE at selected SNOTEL sites.  We focus on analyzing model performance both for **a single SNOTEL site** and **watershed-scale behavior for multiple stations**, with particular attention to the **magnitude and timing of peak SWE**.   

This notebook requires ParFlow-CONUS output, SNOTEL data, and metadata CSVs that are created in the `01_HydroData_collection.ipynb` notebook.

## 1. Prepare the Python Environment

Ensure that the `nwm_env` conda environment is selected as your Jupyter kernel. This environment should already be created if you followed the instructions under section "Creating your HydroLearnEnv Virtual Environment" in the `getting_started.md` file.

Import the libraries needed to run this notebook:

In [ ]:
import os
import sys
import glob
from pathlib import Path

import holoviews as hv
import hvplot.pandas
import geopandas as gpd
import pandas as pd

# Import the Evaluation library from the project root.
sys.path.append(str((Path.cwd().absolute() / "../../../src").resolve()))
from cssi_evaluation.snow import utils
from cssi_evaluation import evaluation_metrics

# import notebook-specific utilities
from utils import nwm_utils

hv.extension('bokeh')


%load_ext autoreload
%autoreload 2

## 2. Build the Input DataFrame

This workflow requires us to organize our observed and modeled snow water equivilent into a single Pandas DataFrame. We'll do this by reading the CSVs created in the `01_HydroData_collection.ipynb` notebook. 

**Quick Reminder:** Observed SWE is derived from SNOTEL measurements, while modeled SWE is extracted from ParFlow-CONUS at corresponding station locations. All time series here are evaluated at a daily resolution.


In [ ]:
# Start and end times of a water year (note that this code currently works for one water year)
StartDate = '2018-10-01'
EndDate = '2020-09-30'

Read in existing CSV files of observations and model outputs for one HUC-08

In [ ]:
# Observations 
obs_df = pd.read_csv(os.path.join('obs_outputs_PF', 'df_East-Taylor_14020001_SNOTEL.csv'))
obs_df = obs_df.set_index("date")

# Observations metadata 
metadata_df = pd.read_csv(os.path.join('obs_outputs_PF', 'df_East-Taylor_14020001_SNOTEL_metadata.csv'))

# Model outputs
mod_df = pd.read_csv(os.path.join('mod_outputs_PF', 'df_East-Taylor_14020001_PFCONUS1.csv'))
mod_df = mod_df.set_index("date")

In [ ]:
# Combine these into a single dataframe
combined_df = pd.concat([obs_df, mod_df], axis=1)
combined_df.index = pd.to_datetime(combined_df.index)

combined_df

## 3. Spatial Mapping of the SNOTEL sites  
Before evaluating model performance, we plot the GIS data associated with the records in the combined DataFrame. The map below shows the SNOTEL stations included in the evaluation dataset, along with the watershed boundary used for the model simulations. Hover over the pins to see the site names.  

We also print a table of the SNOTEL site metadata to help with the single site selection.

In [ ]:
# Path to the watershed shapefile
watershed = "./domain_data/East-Taylor_14020001.shp"
watershed_gdf = gpd.read_file(watershed).to_crs(epsg=4326)

# Create GeoDataFrame of all available stations
filtered_all_stations_gdf = gpd.GeoDataFrame(
    metadata_df,
    geometry=gpd.points_from_xy(
        metadata_df.longitude,
        metadata_df.latitude
    ),
    crs="EPSG:4326"
)
print("Sites CRS:", filtered_all_stations_gdf.crs)

# Combine watershed polygons into one geometry
watershed_union = watershed_gdf.geometry.unary_union

# Filter stations that fall within the watershed
sites_in_watershed = filtered_all_stations_gdf[
    filtered_all_stations_gdf.geometry.within(watershed_union)
].copy()

sites_in_watershed.reset_index(drop=True, inplace=True)

print(f"Total sites in watershed: {len(sites_in_watershed)}")

m = nwm_utils.plot_sites_within_domain(sites_in_watershed, watershed_gdf, zoom_start=9)
m

In [ ]:
sites_in_watershed

## 4. Compare Modeled and Observed SWE Timeseries at a Single Site

Once we have both observation data and modeling outpus, it's important to evaluate how well the model reproduces observed data. The following plots are simple timeseries comparisons of **modeled vs. observed** SWE. These types of plots provide a straight-forward visual of how well the observations and simulations agree and are a great start for assessing general model performance.  

📊 We include two figures:

1. **Time Series Overlay:** Plots the observed and modeled values together over time. This helps identify:
   - Periods of systematic bias
   - Timing differences in peaks and lows
   - General agreement in trends

2. **Scatter Plot with 1:1 Line:** Plots each modeled value against its corresponding observed value. This highlights:
   - Accuracy across the full range of values
   - Over- or under-prediction patterns
   - Outliers or extreme events

Review the sites within the watershed from the interactive map above and click on the markers to view the site name and code. Recall, we also printed out the site metadata for all sites within the watershed, which contains the 3-letter site codes.

✏️ Once you’ve identified the site of interest, **enter its site code in the next code cell for `my_site_code`**:   

In [ ]:
# choose a site of interest within the watershed
my_site_code = '380:CO:'

############################ THIS BELOW DOESNT WORK BECAUSE CODE IS NOT COMPLETE
# filter to only that site
sites_in_watershed[sites_in_watershed['site_id']==my_site_code]

In [ ]:
nwm_utils.comparison_plots(combined_df, f'{my_site_code}SNTL', f'{my_site_code}PFCONUS1')

To move beyond an overall summary of daily performance, we replot the modeled vs. observed SWE scatter while highlighting specific months with a distinct color. This gives us more information about the **seasonal model performance**.  

Let's customize the scatter plot by allowing you to highlight specific months with a distinct color. The selected months will appear in one color, while all other months will appear in a different color. This customization reveals whether there are **seasonal patterns** in the relationship between observed and modeled SWE, allowing us to distinguish model behavior during the key snowpack phases of <span style="color: teal;">accumulation</span> and <span style="color: tomato;">ablation (melt)</span>. Identifying these patterns is important for diagnosing the model’s strengths and limitations during different parts of the snow season.

You can change the list of highlighted months (for example, October–December for early accumulation or March–May for spring melt) to explore in the scatter plot how model performance varies across different parts of the snow season. This seasonal perspective motivates the _peak SWE analysis_ that follows.

##### 📊 For this example, let's highlight the _early snow accumulation period_ of October - January:

In [ ]:
combined_df['month'] = combined_df.index.month

plot = nwm_utils.plot_custom_scatter_SWE(combined_df, f'{my_site_code}SNTL', f'{my_site_code}PFCONUS1',
                                        highlight_months=[10, 11, 12, 1])
plot 

<div style="color:black; background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
<h4>🧠 Reflect</h4>
<p>What does this plot tell us about how well the model performs during the early snow accumulation period at this site?<br>
HINT: How close are the <span style="color: teal;">green points</span> to the 1:1 line?</p>
</div>

## 5. Peak SWE Evaluation at the Watershed Scale  
As we saw in the previous section, how well a model matches observations can differ greatly throughout the year. The following section focuses on **peak SWE** (or maximum SWE) analysis. 

**Peak SWE is a key diagnostic for snow-dominated hydrologic systems** because it represents the maximum amount of liquid water stored in the snowpack before the spring melt. Evaluating both the magnitude (quantity) and timing (date) of peak SWE provides insight into whether the model is accurately representing snow accumulation and seasonal energy balance.  

Errors in peak SWE can have important hydrologic consequences, as peak accumulation strongly influences:
- The volume of water available for spring runoff
- The timing of streamflow peaks
- Soil moisture recharge and groundwater contributions

<div style="padding: 10px;">
  <img src="img/peak_swe_example.png" style="width:85%;">
</div>  

_Example daily SWE at a single site, showing two important periods in snow processes: accumulation (before peak) and ablation (after peak). The vertical line marks peak SWE._

### 5.1 Comparing Modeled and Observed Peak SWE at All Sites in the Watershed
In this section, we evaluate observed and modeled peak SWE for all stations within our watershed and for all years selected in the `StartDate` and `EndDate` above. 

#### 📋 Modeled SWE on the Date of Observed Peak SWE (magnitude)  
This comparison evaluates the modeled SWE on the **specific day when observed SWE reaches its maximum.** By fixing the timing to the observed peak, this comparison isolates errors in SWE magnitude.  
It answers the question: *How much SWE does the model simulate on the day the observed snowpack reaches its maximum?*

In [ ]:
# isolate the columns associated with observations and model predictions.
# these will be inputs to our same-day comparison function.
obs_cols = sorted([col for col in combined_df.columns if col.endswith('SNTL')])
mod_cols = sorted([col for col in combined_df.columns if col.endswith('PFCONUS1')])

In [ ]:
# compute the same-day SWE comparison during the observed peak SWE for each of the observation and modeled sites.
df_observed_peak = utils.modeled_swe_at_observed_peak(combined_df, obs_cols, mod_cols)
df_observed_peak

##### 📊 Visualize the amount of SWE on **the day of observed peak SWE occurs** for both the model and observations at each station

In [ ]:
# Rearrange the dataframe to long format for easier plotting
df_long = (
    df_observed_peak
    .reset_index()  
    .melt(
        id_vars=['Station', 'Water_Year', 'date'],
        value_vars=['Observed', 'Modeled'],
        var_name='Source',
        value_name='SWE'
    )
)
# Create scatter plot of observed and modeled SWE on the day of observed peak SWE
scatter_obs_peak = df_long.hvplot.scatter(
    x='Station',
    y='SWE',
    by='Source',  # Observed vs Modeled
    ylabel='SWE on Observed Peak Day (mm)',
    title='Observed and Modeled SWE on the Day of Observed Peak SWE',
    size=70,
    width=700,
    height=450,
    alpha=0.8,
    hover_cols=['Water_Year'],
    rot=45
)

# Customize the scatter plot appearance
scatter_by_station = (
    scatter_obs_peak  
    .opts(legend_position='top_right')
)

scatter_by_station

#### 📋 Modeled vs Observed Peak SWE Comparison (timing & magnitude)  
This comparison evaluates the modeled and observed peak SWE values and their corresponding dates independently. Unlike the previous comparison that fixed the timing to the observed peak swe, this analysis shows the actual days of modeled and observed peak SWE, which may occur on different dates. As a result, it captures errors in both **peak SWE magnitude** and **peak timing**.

In [ ]:
# compute the different-day SWE comparison for each of the observed and modeled sites.
df_both_peak = utils.modeled_vs_observed_peak_swe(combined_df, obs_cols, mod_cols)
df_both_peak

##### 📊 Visualize the quantity of peak SWE for both the model and observations at each station

In [ ]:
### NEED TO DECIDE HOW TO FORMAT THIS PLOT AND IF WE WANT TO HAVE THE "SAME_DAY" PLOT

# Create the scatter plot
scatter_plot_both_peak = df_both_peak.hvplot.scatter(
    x='Observed',
    y='Modeled',
    xlabel='Observed SWE (mm)',
    ylabel='Modeled SWE (mm)',
    title='Modeled vs. Observed Peak SWE',
    size=35,
    width=500,
    height=400,
    color='#E69F00',
    hover_cols=['Station', 'Water_Year']
)#.relabel('Peak SWE')

# Add 1:1 line (perfect match line)
swe_max = df_both_peak[['Observed', 'Modeled']].max().max()

one_to_one_line = hv.Curve(([0, swe_max], [0, swe_max])).opts(
    color='gray',
    line_dash='dashed',
    line_width=1,
).relabel('1:1 Line')

# Combine scatter plot and 1:1 line into an Overlay
scatter_with_line = (scatter_plot_both_peak * one_to_one_line).opts( #scatter_plot_obs_peak * 
    legend_position='bottom_right'
)

scatter_with_line

### 5.2 Visualizing Model Error for Peak SWE

The previous scatter plots indicate that the modeled and observed peak SWE magnitude and timing don't always align. Next, we plot the degree to which   

The previous scatter plots highlight differences between modeled and observed peak SWE timing and magnitude, but interpreting these variations can be challenging when comparing modeled and observed values directly. To make these differences more explicit, we compute errors in both peak timing and peak SWE magnitude and visualize them directly. This approach clarifies both the direction and magnitude of model bias and facilitates comparison across stations and water years.

First, add columns `Peak_Date_Diff_Days` and `Peak_SWE_Diff` to the DataFrame `df_both_peak` for computed difference in peak SWE date difference and peak SWE quantity between modeled and observed:

In [ ]:
# Compute the difference in peak SWE days and peak SWE amounts between modeled and observed
df_both_peak['Peak_Date_Diff_Days'] = (df_both_peak['Modeled_Date'] - 
                                      df_both_peak['Observed_Date']).dt.days
df_both_peak['Peak_SWE_Diff'] = (df_both_peak['Modeled'] - 
                           df_both_peak['Observed'])

In [ ]:
df_both_peak

##### 📊 Visualize the error between the modeled and observed peak SWE  

In [ ]:
# Filter to separate each water year
year1 = df_both_peak[df_both_peak['Water_Year'] == df_both_peak['Water_Year'].min()]
year2 = df_both_peak[df_both_peak['Water_Year'] == df_both_peak['Water_Year'].max()]

bar1 = year1.hvplot.bar(
    x='Station',
    y='Peak_Date_Diff_Days',
    rot=45,
    ylabel='Date Difference (days)',
    title=f'Peak SWE Date Difference {year1["Water_Year"].iloc[0]} (model - obs)',
    width=400,
    height=400,
    color='Peak_Date_Diff_Days',
    hover_cols=['Modeled', 'Observed']
)
bar2 = year1.hvplot.bar(
    x='Station',
    y='Peak_SWE_Diff',
    rot=45,
    ylabel='SWE Difference (m)',
    title=f'Peak SWE Difference {year1["Water_Year"].iloc[0]} (model - obs)',
    width=400,
    height=400,
    color='Peak_SWE_Diff',
    hover_cols=['Modeled', 'Observed']
)

# Combine side by side
layout = (bar1 + bar2)
layout.opts(shared_axes=False)

The left panel shows the timing error (date difference) and the right panel the magnitude error (SWE difference). When we computed the difference in date and SWE quantity above, we took `modeled - observed` so:  

| | DATE OF PEAK SWE | PEAK SWE QUANTITY |
|---|---|---|
| + Positive Values | modeled AFTER observed | modeled GREATER THAN observed |
| - Negative Values | modeled BEFORE observed | modeled LESS THAN observed |  

<div style="color:black; background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
<h4>🧠 Reflect</h4>
<p>Looking at the two plots, what could be some reasons for the model having simulated peak SWE both earlier and less than the observed peak SWE? Perhaps try changing the <code>my_site_code</code> from earlier in the notebook to rerun <code>nwm_utils.comparison_plots()</code> to see the timeseries for a different station to look at the peak magnitude and timing. 

<br>What happens if you change the year that is plotted? <br>✏️ Try modifying the bar plot code from <code>bar1 = year1.hvplot.bar</code> to <code>bar1 = year2.hvplot.bar</code>. Don't forget to change the title!</p>
</div>

#### 📊 Next, we combine the timing and magnitude errors and plot them together for each station.

In [ ]:


scatter = df_both_peak.hvplot.scatter(
    x='Peak_Date_Diff_Days',
    y='Peak_SWE_Diff',
    by='Station', # Water_Year
    xlabel='Peak SWE Timing Error (days)',
    ylabel='Peak SWE Magnitude Error (mm)',
    title='Peak SWE Timing vs Magnitude Error',
    size=80,
    width=600,
    height=400,
    hover_cols=['Water_Year']
)

# Add reference lines
vline = hv.VLine(0).opts(color='gray', line_dash='dashed')
hline = hv.HLine(0).opts(color='gray', line_dash='dashed')

(scatter * vline * hline).opts(legend_position='top_left', show_grid=True)


✏️ **Try changing how we view this plot.**  
We can modify a line in the section of code from `by='Station'` to `by='Water_Year'` to better visualize the errors in the different Water Years.  
Are there any patterns that jump out? Which year was modeled peak SWE consistently less than observed peak SWE? 

## 6. Compute and Statistics and Error Metrics  
The previous section visualized when and where modeled SWE differs from observations, both in terms of peak SWE timing and magnitude. However, visual inspection alone makes it difficult to compare performance across sites or to summarize model behavior in a consistent or quantifiable way. In this section, we compute commonly used statistical error metrics to quantify model performance, allowing us to objectively assess bias, error magnitude, and variability for sites within the watershed.  

Proposed outline (DTK, Jan 2026):
- Summary metrics at a station
- Summary metrics at all stations within the watershed
- Combined timing and magnitude for all stations within the watershed (Condon metric)
- Focus on timing: summary statistics for single station for accumulation & ablation periods (using the new wrapper: `nwm_utils.compute_stats_period()`)
- Melt period statistics

In [ ]:
nwm_utils.compute_stats(combined_df, f'CCSS_{my_site_code}_swe_m', f'NWM_{my_site_code}_swe_m')

Pearson and Spearman correlations are both close to 1, suggesting a strong relationship between observed and modeled SWE. As shown on the timeseries plot, this strong correlation alone does not indicate a "good" model. For example, it does not guarantee accurate timing of key events, such as peak SWE or melt onset. Let's compare these as well. The following code uses `report_max_dates_and_values` function to identify the peak SWE value and the date it occurs for both the observed (CCSS) and modeled (NWM) datasets. 

<div style="color:black; background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
<h4>🧠 Reflect</h4>
<p>You now have several performance metrics: Bias, Pearson Correlation, Spearman Correlation, NSE, and KGE. If you had to pick just one metric to summarize model performance, which would you choose—and why? As you review the results, compare the peak flow amounts and the timing of snowmelt onset. Do you see any significant differences? Which dataset indicates an earlier melt?</p>
</div>

In [ ]:
summary_table = nwm_utils.report_max_dates_and_values(combined_df, f'CCSS_{my_site_code}_swe_m', f'NWM_{my_site_code}_swe_m')
summary_table

### Summary Metrics at Multiple Sites

In [ ]:
site_codes = ['DAN', 'HRS', 'KIB', 'PDS', 'SLI', 'TUM', 'WHW']

rows = []

for site in site_codes:
    obs_col = f'CCSS_{site}_swe_m'
    mod_col = f'NWM_{site}_swe_m'

    stats_table = nwm_utils.compute_stats(combined_df, obs_col, mod_col)

    rows.append({
        'Station': site,
        'Mean_Obs': stats_table.loc['observed', 'Mean'],
        'Mean_Mod': stats_table.loc['modeled', 'Mean'],
        'Bias_m': stats_table.loc['Bias (Modeled - Observed)', 'Mean'],
        'Pearson_r': stats_table.loc['Pearson Correlation', 'Mean'],
        'Spearman_r': stats_table.loc['Spearman Correlation', 'Mean'],
        'NSE': stats_table.loc['Nash-Sutcliffe Efficiency (NSE)', 'Mean'],
        'KGE': stats_table.loc['Kling-Gupta Efficiency (KGE)', 'Mean']
    })

stats_AllStations = pd.DataFrame(rows)

print('All Stations Statistics Summary:')
stats_AllStations

In [ ]:
stats_AllStations.hvplot.bar(
    x='Station',
    y='NSE',
    rot=45,
    ylabel='Nash–Sutcliffe Efficiency',
    title='NSE by Station',
    height=400,
    width=600,
    bar_width=0.5
)


In [ ]:
stats_summary.hvplot.scatter(
    x='Station',
    y='Bias_m',
    size=100,
    rot=45,
    ylabel='Bias (m)',
    title='Mean SWE Bias by Station'
)


### Combine Magnitude (absolute relative bias) and Timing (Spearman's rho) metrics using the Condon metric (and with all stations, a Condon diagram)

In [ ]:
bias1 = evaluation_metrics.bias(combined_df.CCSS_TUM_swe_m, combined_df.NWM_TUM_swe_m)
bias1

In [ ]:
abs_bias = evaluation_metrics.absolute_relative_bias(combined_df.CCSS_TUM_swe_m, combined_df.NWM_TUM_swe_m)
abs_bias

In [ ]:
srho = evaluation_metrics.spearman_rank(combined_df.CCSS_TUM_swe_m, combined_df.NWM_TUM_swe_m)
srho

In [ ]:
evaluation_metrics.condon(abs_bias, srho)

<div style="color:black; background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
  <h4>🧠 Reflect</h4>
  <p>
    What is the modeled SWE on the date when the observed SWE reaches its peak?<br>
    ✏️ Use the code snippet below to find the answer.
  </p>
  <pre><code class="language-python">
  
    # Find date of the peak SWE from observed data
    date_obs_max = combined_df['CCSS_HRS_swe_m'].idxmax()

    # Get corresponding value of modeled data on that date
    value_mod_at_max_obs = combined_df.loc[date_obs_max, 'NWM_HRS_swe_m']
  </code></pre>
</div>


### Focus on Timing: Melt Period Metrics
Compare the average melt rate over the full melt period. 

The following function computes the melt period length by identifying the first date after the peak SWE when SWE drops to zero and remains at zero for at least (`min_zero_days`) consecutive days. This is used to define the end of the melt period. Finally, the function calculates the average melt rate, which represents the rate at which snow disappeared, expressed in meters per day, over the full melt period.

In [ ]:
melt_stats_df = utils.compute_melt_period_statistics(combined_df)
melt_stats_df.head()
melt_stats_df

In [ ]:
observed_melt_period = nwm_utils.compute_melt_period(combined_df[f'CCSS_{my_site_code}_swe_m'])
observed_melt_period

In [ ]:
modeled_melt_period = nwm_utils.compute_melt_period(combined_df[f'NWM_{my_site_code}_swe_m'])
modeled_melt_period

In [ ]:
accum_months = [10, 11, 12, 1, 2, 3]
ablation_months = [4, 5, 6]

accum_stats = nwm_utils.compute_stats_period(
    combined_df,
    f'CCSS_{my_site_code}_swe_m',
    f'NWM_{my_site_code}_swe_m',
    accum_months
)

accum_stats

In [ ]:

ablation_stats = nwm_utils.compute_stats_period(
    combined_df,
    f'CCSS_{my_site_code}_swe_m',
    f'NWM_{my_site_code}_swe_m',
    ablation_months
)

ablation_stats

<div style="color:black; background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
  <h4>🧠 Reflect</h4>
  <p>
    If you recall from earlier, we plotted the timeseries of out selected station. Replot it below. Do the metrics make sense given the visual comparison between modeled and observed? For example, when you look at the timeseries, is the model consistently predicting SWE to be higher or lower than observations? Does this align with the <b>Bias</b> sign (+ or -)?
  </p>
</div> 

In [ ]:
nwm_utils.comparison_plots(combined_df, f'CCSS_{my_site_code}_swe_m', f'NWM_{my_site_code}_swe_m')
